# nnUNet Baseline — BraTS 2024 MEN-RT

**Setup:** 50 training cases (5-fold CV) · 10 held-out test cases · 50 epochs/fold  
**Evaluation:** Dice, HD95, Sensitivity, Precision — raw vs postprocessed predictions

**Before running:**
1. Right panel → Add Data → attach BraTS-MEN-RT dataset
2. Right panel → Internet → ON
3. Right panel → Accelerator → GPU T4 x2
4. Run cells top to bottom

**Resume after disconnect:** re-run Cell 3 (restores env), then re-run the interrupted cell with `--continue` appended.

```
Cell  1 : GPU check
Cell  2 : Install nnunet==1.7.1
Cell  3 : Clone repo + setup env
Cell  4 : Verify dataset
Cell  5 : Prepare dataset (50 train + 10 held-out test)
Cell  6 : Plan & preprocess  ← nnUNet auto-configures patch/batch
Cell  7 : Create stratified splits
Cell  8 : Install custom trainer
Cell  9 : Train Fold 0  (50 epochs)  ← run this first
Cell 10 : Quick Fold-0 metrics check
Cell 11-14 : Train Folds 1-4
Cell 15 : Full CV metrics table
Cell 16 : Inference on held-out test set
Cell 17 : Evaluate test set (raw vs postprocessed)
Cell 18 : All plots
Cell 19 : Training curves
Cell 20 : Artifacts listing
```

In [ ]:
# Cell 1: GPU check — must show a T4 before proceeding
!nvidia-smi

In [ ]:
# Cell 2: Install dependencies
!pip install -q nnunet==1.7.1
!pip install -q pyyaml simpleitk scipy
print('Installation complete.')

In [ ]:
# Cell 3: Clone / pull repo and set working directory
# Re-run this cell any time the kernel restarts.
import os

REPO_DIR = '/kaggle/working/brats-menrt-mednext-thesis'
REPO_URL = 'https://github.com/maryampervaiz-alt/brats-menrt-mednext-thesis.git'
CONFIG   = 'configs/nnunet_baseline.yaml'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest …')
    os.system(f'cd {REPO_DIR} && git pull origin main')
else:
    print('Cloning repo …')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# ── Environment variables (inherited by all ! shell commands) ────────────
os.environ['nnUNet_raw_data_base'] = '/kaggle/working/nnUNet_raw_data_base'
os.environ['nnUNet_preprocessed']  = '/kaggle/working/nnUNet_preprocessed'
os.environ['RESULTS_FOLDER']       = '/kaggle/working/nnUNet_results'
os.environ['NNUNET_MAX_EPOCHS']    = '50'
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'
print('Environment variables set.')
!ls

In [ ]:
# Cell 4: Verify dataset is accessible
TRAIN_ROOT = '/kaggle/input/datasets/maryampervaiz24029/brats-men-rt/BraTS2024-MEN-RT-TrainingData/BraTS-MEN-RT-Train-v2'

if os.path.exists(TRAIN_ROOT):
    n = len([d for d in os.listdir(TRAIN_ROOT) if os.path.isdir(os.path.join(TRAIN_ROOT, d))])
    print(f'Train root: OK  ({n} cases)')
    print(f'Path: {TRAIN_ROOT}')
else:
    print(f'NOT FOUND: {TRAIN_ROOT}')
    print('Run: !find /kaggle/input -maxdepth 5 -type d  to find the correct path')
    raise FileNotFoundError('Dataset not found. Attach it via Kaggle → Add Data.')

In [ ]:
# Cell 5: Prepare dataset
# Selects 50 stratified training cases + 10 held-out test cases from the labeled pool.
# The 10 test cases are NEVER used during training or cross-validation.
# Their labels are saved in labelsTs/ for evaluation.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode prepare

In [ ]:
# Cell 6: Plan and preprocess
# nnUNet automatically configures patch_size and batch_size from the dataset.
# DO NOT override these values — auto-configuration is nnUNet's core strength.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode preprocess

In [ ]:
# Cell 7: Create stratified 5-fold splits
# Balances tumour volume distribution across all 5 folds.
# Saves splits_final.pkl — nnUNet reads this during training.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode make-splits

In [ ]:
# Cell 8: Install custom trainer
# Creates nnUNetTrainerV2_MENRT — reads max epochs from NNUNET_MAX_EPOCHS env var.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode install-trainer

## Training

**nnUNet checkpoint files per fold:**
- `model_best.model` — saved whenever validation Dice improves
- `model_latest.model` — saved every 50 epochs
- `model_final_checkpoint.model` — saved at end of training

**Strategy:** Run Fold 0 first (Cell 9), check metrics (Cell 10), then run Folds 1–4.  
**Resume after disconnect:** re-run Cell 3, then re-run the fold cell with `--continue` added.

In [ ]:
# Cell 9: Train Fold 0 — 50 epochs
# Primary fold. Evaluate on Cell 10 before running other folds.
# To resume if interrupted: add --continue-training
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode train --fold 0 --max-epochs 50

In [ ]:
# Cell 10: Quick Fold-0 metrics check
# Read nnUNet's automatically generated summary.json after Fold 0 completes.
import json, os

import yaml
with open(f'{REPO_DIR}/{CONFIG}') as f:
    _cfg = yaml.safe_load(f)['nnunet_baseline']

summary_path = os.path.join(
    _cfg['results_folder'], 'nnUNet', _cfg['network'], _cfg['task_name'],
    f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}",
    'fold_0', 'validation_raw', 'summary.json'
)

if os.path.exists(summary_path):
    with open(summary_path) as f:
        s = json.load(f)
    mean = s['results']['mean']['1']
    print('=' * 45)
    print('  Fold 0 Validation Metrics (nnUNet CV)')
    print('=' * 45)
    print(f"  Dice (DSC)  : {mean.get('Dice', 'N/A'):.4f}")
    print(f"  HD95 (mm)   : {mean.get('Hausdorff Distance 95', 'N/A'):.2f}")
    print(f"  Sensitivity : {mean.get('Recall', 'N/A'):.4f}")
    print(f"  Precision   : {mean.get('Precision', 'N/A'):.4f}")
    print('=' * 45)
    print(f'  n_val_cases : {len(s["results"]["all"])}')
else:
    print('summary.json not found — Fold 0 may not be complete yet.')
    print(f'Expected: {summary_path}')

In [ ]:
# Cell 11: Train Fold 1 — 50 epochs
# To resume: add --continue-training
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode train --fold 1 --max-epochs 50

In [ ]:
# Cell 12: Train Fold 2 — 50 epochs
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode train --fold 2 --max-epochs 50

In [ ]:
# Cell 13: Train Fold 3 — 50 epochs
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode train --fold 3 --max-epochs 50

In [ ]:
# Cell 14: Train Fold 4 — 50 epochs
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode train --fold 4 --max-epochs 50

## Evaluation

Two types of evaluation:

1. **Cross-validation (CV) metrics** — from nnUNet's `summary.json` per fold.  
   These are validation cases *within* the 50 training cases (fold held-out).

2. **Held-out test set metrics** — 10 cases *never seen during training*.  
   Ensemble of all 5 folds. Compared before and after nnUNet postprocessing.

**Why held-out test?**  
CV metrics can be optimistic (model selected on validation loss).  
The held-out test set gives an unbiased estimate of generalisation.

In [ ]:
# Cell 15: Full CV metrics table (all completed folds)
import json, os
import numpy as np

import yaml
with open(f'{REPO_DIR}/{CONFIG}') as f:
    _cfg = yaml.safe_load(f)['nnunet_baseline']

trainer_root = os.path.join(
    _cfg['results_folder'], 'nnUNet', _cfg['network'], _cfg['task_name'],
    f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}"
)

fold_data = []
for fold in _cfg.get('folds', [0,1,2,3,4]):
    sp = os.path.join(trainer_root, f'fold_{fold}', 'validation_raw', 'summary.json')
    if not os.path.exists(sp):
        continue
    with open(sp) as f:
        s = json.load(f)
    m = s['results']['mean']['1']
    fold_data.append({
        'fold': fold,
        'dice': m.get('Dice', float('nan')),
        'hd95': m.get('Hausdorff Distance 95', float('nan')),
        'sens': m.get('Recall', float('nan')),
        'prec': m.get('Precision', float('nan')),
        'n':    len(s['results']['all'])
    })

if fold_data:
    print(f"{'Fold':>5}  {'Dice':>8}  {'HD95':>8}  {'Sens':>8}  {'Prec':>8}  {'N':>4}")
    print('-' * 52)
    for d in fold_data:
        print(f"{d['fold']:>5}  {d['dice']:>8.4f}  {d['hd95']:>8.2f}  {d['sens']:>8.4f}  {d['prec']:>8.4f}  {d['n']:>4}")
    print('-' * 52)
    print(f"{'MEAN':>5}  {np.nanmean([d['dice'] for d in fold_data]):>8.4f}  "
          f"{np.nanmean([d['hd95'] for d in fold_data]):>8.2f}  "
          f"{np.nanmean([d['sens'] for d in fold_data]):>8.4f}  "
          f"{np.nanmean([d['prec'] for d in fold_data]):>8.4f}")
    print(f"{'STD':>5}  {np.nanstd([d['dice'] for d in fold_data]):>8.4f}  "
          f"{np.nanstd([d['hd95'] for d in fold_data]):>8.2f}  "
          f"{np.nanstd([d['sens'] for d in fold_data]):>8.4f}  "
          f"{np.nanstd([d['prec'] for d in fold_data]):>8.4f}")
    print(f'\nCompleted folds: {[d["fold"] for d in fold_data]}')
else:
    print('No completed folds found.')

In [ ]:
# Cell 16: Inference on held-out test set (ensemble of all 5 folds)
# Runs TWICE:
#   1. Default (postprocessed) — nnUNet applies learned postprocessing
#   2. Raw (--disable_postprocessing) — pure network output
# This lets us compare metrics before and after postprocessing.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode predict-testset

In [ ]:
# Cell 17: Evaluate held-out test set predictions
# Computes Dice, HD95, Sensitivity, Precision per case
# for both raw and postprocessed predictions.
# Prints a comparison table showing whether postprocessing helps.
os.chdir(REPO_DIR)
!python scripts/run_nnunet.py --config {CONFIG} --mode evaluate-testset

In [ ]:
# Cell 18: Generate all plots
# CV plots : fold bar, Dice boxplot, HD95 boxplot, Dice vs HD95 scatter
# Test plots: raw vs postprocessed bar, test Dice boxplot
# Also saves cv_metrics_table.csv for thesis
os.chdir(REPO_DIR)
!python scripts/plot_nnunet_results.py --config {CONFIG} --dpi 200

In [ ]:
# Cell 19: Display training curves inline
from IPython.display import Image, display
import yaml, os

with open(f'{REPO_DIR}/{CONFIG}') as f:
    _cfg = yaml.safe_load(f)['nnunet_baseline']

trainer_root = os.path.join(
    _cfg['results_folder'], 'nnUNet', _cfg['network'], _cfg['task_name'],
    f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}"
)

for fold in _cfg.get('folds', [0,1,2,3,4]):
    png = os.path.join(trainer_root, f'fold_{fold}', 'progress.png')
    if os.path.exists(png):
        print(f'\nFold {fold} — training loss + validation Dice:')
        display(Image(filename=png, width=680))
    else:
        print(f'Fold {fold}: progress.png not found yet.')

In [ ]:
# Cell 20: List all saved artifacts and checkpoints
import yaml, os

with open(f'{REPO_DIR}/{CONFIG}') as f:
    _cfg = yaml.safe_load(f)['nnunet_baseline']

trainer_root = os.path.join(
    _cfg['results_folder'], 'nnUNet', _cfg['network'], _cfg['task_name'],
    f"{_cfg['trainer_name']}__{_cfg['plans_identifier']}"
)
artifacts = os.path.join(_cfg['results_folder'], 'menrt_artifacts')

print('=== Checkpoints ===')
for fold in _cfg.get('folds', [0,1,2,3,4]):
    fold_dir = os.path.join(trainer_root, f'fold_{fold}')
    for ckpt in ['model_best.model', 'model_latest.model', 'model_final_checkpoint.model']:
        p = os.path.join(fold_dir, ckpt)
        print(f'  fold_{fold}/{ckpt}: {"OK" if os.path.exists(p) else "missing"}')

print('\n=== Reports ===')
reports = os.path.join(artifacts, 'reports')
if os.path.exists(reports):
    for fn in sorted(os.listdir(reports)):
        fp = os.path.join(reports, fn)
        print(f'  {fn}  ({os.path.getsize(fp)/1024:.1f} KB)')

print('\n=== Figures ===')
figs = os.path.join(artifacts, 'figures')
if os.path.exists(figs):
    for fn in sorted(os.listdir(figs)):
        fp = os.path.join(figs, fn)
        print(f'  {fn}  ({os.path.getsize(fp)/1024:.1f} KB)')

print(f'\nAll outputs under: /kaggle/working/')